# ToothWise: image-level evaluation (one click)

Scores YOLOv8, YOLOv12 and the ensemble **per photo**, giving accuracy, precision, recall,
specificity, F1 and AUROC per condition. See `evaluate_image_level.py` for details.

**How to use**
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Fill in **Step 1** below (where your dataset is).
3. **Runtime → Run all**. Answer any prompts that appear (Google Drive permission, GitHub token).
4. At the end, the results table is shown and the files download automatically.

## Step 1: Where is your dataset? (edit this cell)

In [ ]:
# Choose ONE:
#   "folder"   - dataset already unzipped in THIS Colab session (e.g. /content/merged_dataset)
#   "drive"    - a .zip in your Google Drive
#   "roboflow" - download from Roboflow
DATASET_SOURCE = "folder"

# --- if "folder": the dataset folder ---
DATASET_FOLDER = "/content/merged_dataset"

# --- if "drive": path of the dataset zip inside your Google Drive ---
DRIVE_ZIP = "/content/drive/MyDrive/merged_dataset.zip"

# --- if "roboflow": copy these from Roboflow's "show download code" ---
ROBOFLOW_API_KEY   = ""
ROBOFLOW_WORKSPACE = ""
ROBOFLOW_PROJECT   = ""
ROBOFLOW_VERSION   = 1

# Leave as None to auto-detect the dataset yaml (data_5class.yaml / data.yaml)
DATA_YAML = None

# "test" is what goes in the paper. It needs a val/valid split too (used to tune thresholds).
SPLIT = "test"

## Step 2: Check the GPU

In [ ]:
import subprocess
try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True).stdout)
except Exception:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all again.")

## Step 3: Get the scripts and model weights from GitHub

In [ ]:
import os, subprocess, getpass
REPO, BRANCH, DEST = "zenzenzh/toothwise", "ccr-e18dfb7b-jk2583", "/content/toothwise"

if not os.path.isdir(DEST):
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                        f"https://github.com/{REPO}.git", DEST],
                       capture_output=True, text=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if r.returncode != 0:
        print("Repo is private. Paste a GitHub token (Settings -> Developer settings -> "
              "Personal access tokens -> classic, 'repo' scope). It is not saved.")
        token = getpass.getpass("GitHub token: ")
        r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                            f"https://{token}@github.com/{REPO}.git", DEST],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise SystemExit("Clone failed - check the token.\n" + r.stderr.replace(token, "***"))
        # don't leave the token in .git/config
        subprocess.run(["git", "-C", DEST, "remote", "set-url", "origin",
                        f"https://github.com/{REPO}.git"])
        del token

WEIGHTS = f"{DEST}/backend/models_weights"
V8, V12 = f"{WEIGHTS}/yolov8_dental.pt", f"{WEIGHTS}/yolov12_dental.pt"
for p in (V8, V12):
    print(("OK      " if os.path.isfile(p) else "MISSING ") + p)

## Step 4: Install Ultralytics

In [ ]:
!pip install -q 'ultralytics>=8.3' roboflow

## Step 5: Get the dataset

In [ ]:
import glob, zipfile
DATA_DIR = "/content/dataset"

if DATASET_SOURCE == "folder":
    if not os.path.isdir(DATASET_FOLDER):
        raise SystemExit(f"{DATASET_FOLDER} does not exist in this Colab session. Files in /content "
                         "are lost when the runtime restarts, and each notebook has its own runtime. "
                         "Re-create the folder here, or use the 'drive' option.")
    DATA_DIR = DATASET_FOLDER
elif DATASET_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isfile(DRIVE_ZIP):
        zips = glob.glob("/content/drive/MyDrive/**/*.zip", recursive=True)[:30]
        raise SystemExit(f"Not found: {DRIVE_ZIP}\nZips in your Drive:\n" + "\n".join(zips))
    if not os.path.isdir(DATA_DIR):
        with zipfile.ZipFile(DRIVE_ZIP) as z:
            z.extractall(DATA_DIR)
elif DATASET_SOURCE == "roboflow":
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    ds = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT) \
           .version(ROBOFLOW_VERSION).download("yolov8", location=DATA_DIR)
else:
    raise SystemExit('DATASET_SOURCE must be "folder", "drive" or "roboflow"')

if DATA_YAML is None:
    found = [p for p in glob.glob(f"{DATA_DIR}/**/*.yaml", recursive=True)
             if "names" in open(p).read()]
    found.sort(key=lambda p: ("5class" not in p, len(p)))  # prefer data_5class.yaml
    if not found:
        raise SystemExit(f"No dataset yaml under {DATA_DIR}")
    DATA_YAML = found[0]
    if len(found) > 1:
        print("Several yamls found, using the first (set DATA_YAML in Step 1 to override):",
              *found, sep="\n  ")
print("Dataset yaml:", DATA_YAML)
print(open(DATA_YAML).read())

## Step 6: Run the evaluation (about 15-30 minutes)

In [ ]:
%cd /content/toothwise/backend/training
!python evaluate_image_level.py --data "$DATA_YAML" --yolov8 "$V8" --yolov12 "$V12" --split "$SPLIT"

## Step 7: Show and download the results

In [ ]:
from IPython.display import Markdown, display
from google.colab import files
display(Markdown(open("image_level_results.md").read()))
files.download("image_level_results.md")
files.download("image_level_results.json")